# 🎬 Checkpoint 6 · Sistemas recomendadores

Construimos tres recomendadores de películas con los metadatos de **The Movies Dataset** (~45,000 películas de MovieLens/TMDB):

1. **Recomendador simple**: un clon del *Top 250 de IMDB* usando su fórmula de calificación ponderada.
2. **Basado en contenido (sinopsis)**: TF-IDF sobre la trama + similitud coseno.
3. **Basado en contenido (metadatos)**: director, 3 actores principales, géneros y palabras clave + `CountVectorizer`.

Y al final aplicamos las mejoras que sugiere la lección: **más peso al director** y **filtro de popularidad**.

> **Datos:** coloca `movies_metadata.csv`, `credits.csv` y `keywords.csv` en la carpeta `./data/`.

In [1]:
import numpy as np
import pandas as pd
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity

pd.set_option('display.max_colwidth', 60)

## 1. Carga de datos

In [2]:
df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
print(df.shape)
df.head(3)

(45466, 24)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_pa...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Co...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his room until...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fa...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an enchanted board...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': '...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collection', 'pos...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'C...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud between next...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for Love.,Grumpier Old Men,False,6.5,92.0


## 2. Recomendador simple — calificación ponderada de IMDB

Usar solo `vote_average` premia a películas con muy pocos votos (un 10 de 3 personas le ganaría a un 8.9 de 10,000).
Por eso usamos la media ponderada de IMDB:

$$\text{WR} = \frac{v}{v+m}\cdot R + \frac{m}{v+m}\cdot C$$

- **v**: votos de la película · **R**: calificación promedio de la película
- **m**: votos mínimos para entrar a la lista (percentil 90) · **C**: calificación promedio de todo el catálogo

Cuantos más votos tiene una película, más pesa su propia calificación (R); con pocos votos, su score se "jala" hacia el promedio general (C).

In [3]:
# Las columnas numéricas vienen como texto en algunas filas corruptas: las forzamos a número
df['vote_count'] = pd.to_numeric(df['vote_count'], errors='coerce')
df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce')

C = df['vote_average'].mean()
m = df['vote_count'].quantile(0.90)
print(f'C (calificación media del catálogo) = {C:.3f}')
print(f'm (votos mínimos, percentil 90)      = {m:.0f}')

C (calificación media del catálogo) = 5.618
m (votos mínimos, percentil 90)      = 160


In [4]:
q_movies = df.copy().loc[df['vote_count'] >= m]
print('Películas que califican:', q_movies.shape[0], 'de', df.shape[0])

Películas que califican: 4555 de 45466


In [5]:
def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v + m) * R) + (m / (m + v) * C)

q_movies['score'] = q_movies.apply(weighted_rating, axis=1)
q_movies = q_movies.sort_values('score', ascending=False)

q_movies[['title', 'vote_count', 'vote_average', 'score']].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


**Lectura del resultado:** la lista coincide en gran parte con el Top 250 real de IMDB (*The Shawshank Redemption*, *The Godfather*, *The Dark Knight*…).
La limitación es obvia: **todos los usuarios ven la misma lista**. No es personalizada.

### ¿Qué tanto cambia m el ranking?
`m` es un hiperparámetro. Probemos el percentil 80 y 95 para ver su efecto:

In [6]:
def top_n(percentil, n=10):
    m_p = df['vote_count'].quantile(percentil)
    q = df.loc[df['vote_count'] >= m_p].copy()
    q['score'] = q.apply(weighted_rating, axis=1, m=m_p, C=C)
    return q.sort_values('score', ascending=False)['title'].head(n).reset_index(drop=True)

pd.DataFrame({f'p{int(p*100)} (m={df["vote_count"].quantile(p):.0f})': top_n(p) for p in (0.80, 0.90, 0.95)})

,p80 (m=50),p90 (m=160),p95 (m=434)
0,Dilwale Dulhania Le Jayenge,The Shawshank Redemption,The Shawshank Redemption
1,The Shawshank Redemption,The Godfather,The Godfather
2,The Godfather,Dilwale Dulhania Le Jayenge,The Dark Knight
3,Your Name.,The Dark Knight,Fight Club
4,The Dark Knight,Fight Club,Pulp Fiction
5,Fight Club,Pulp Fiction,Forrest Gump
6,Pulp Fiction,Schindler's List,Schindler's List
7,Schindler's List,Whiplash,Whiplash
8,Whiplash,Spirited Away,Spirited Away
9,Spirited Away,Life Is Beautiful,The Empire Strikes Back


Con un `m` más alto solo sobreviven películas muy votadas (más "mainstream"); con uno más bajo entran joyas menos conocidas pero con más riesgo de calificaciones infladas.

## 3. Recomendador basado en contenido — sinopsis (TF-IDF)

Ahora queremos: *"si te gustó X, te recomiendo películas parecidas a X"*.
Convertimos cada sinopsis (`overview`) en un vector **TF-IDF**: palabras frecuentes en esa película pero raras en el resto pesan más.

In [7]:
df['overview'] = df['overview'].fillna('')

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['overview'])
print('Matriz TF-IDF:', tfidf_matrix.shape, '(películas × palabras)')
tfidf.get_feature_names_out()[5000:5010]

Matriz TF-IDF: (45466, 75827) (películas × palabras)


array(['avails', 'avaks', 'avalanche', 'avalanches', 'avallone', 'avalon',
       'avant', 'avanthika', 'avanti', 'avaracious'], dtype=object)

### ⚠️ Nota sobre memoria

La lección calcula `linear_kernel(tfidf_matrix, tfidf_matrix)`: una matriz de **45,466 × 45,466** números.
Eso son ~2,000 millones de valores ≈ **16 GB de RAM**, suficiente para tronar el kernel en la mayoría de las laptops:

In [8]:
n = tfidf_matrix.shape[0]
print(f'Matriz completa: {n:,} x {n:,} = {n*n:,} valores ≈ {n*n*8/1e9:.1f} GB en float64')

Matriz completa: 45,466 x 45,466 = 2,067,157,156 valores ≈ 16.5 GB en float64


**Solución:** para recomendar solo necesitamos **la fila de la película consultada**.
Calculamos la similitud de esa película contra todas las demás (1 × 45,466 ≈ 0.4 MB). El resultado es **idéntico** al de la matriz completa.

Como los vectores TF-IDF ya están normalizados, el producto punto (`linear_kernel`) es igual a la similitud coseno y es más rápido.

In [9]:
# Mapeo inverso título -> índice. Si un título se repite (remakes o duplicados del dataset)
# nos quedamos con la primera aparición para que indices[title] siempre sea un número.
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices = indices[~indices.index.duplicated(keep='first')]
indices[:10]

title
Toy Story                      0
Jumanji                        1
Grumpier Old Men               2
Waiting to Exhale              3
Father of the Bride Part II    4
Heat                           5
Sabrina                        6
Tom and Huck                   7
Sudden Death                   8
GoldenEye                      9
dtype: int64

In [10]:
def get_recommendations(title, matriz=tfidf_matrix, data=df, idx_map=None, n=10):
    """Devuelve las n películas más similares a `title`."""
    idx_map = indices if idx_map is None else idx_map
    if title not in idx_map:
        raise KeyError(f'No encontré "{title}" en el catálogo.')
    idx = idx_map[title]

    # Similitud coseno de ESTA película contra todas (equivale a cosine_sim[idx])
    sim = linear_kernel(matriz[idx], matriz).ravel()

    # Ordenar de mayor a menor y quitar la propia película
    orden = [i for i in sim.argsort()[::-1] if i != idx][:n]
    return pd.DataFrame({'title': data['title'].iloc[orden].values,
                         'similitud': sim[orden].round(3)})

get_recommendations('The Dark Knight Rises')

,title,similitud
0,The Dark Knight,0.327
1,Batman Forever,0.316
2,Batman Returns,0.296
3,Batman: Under the Red Hood,0.288
4,Batman,0.271
5,Batman Unmasked: The Psychology of the Dark Knight,0.253
6,Batman Beyond: Return of the Joker,0.234
7,Batman: Year One,0.204
8,"Batman: The Dark Knight Returns, Part 1",0.199
9,Batman: Mask of the Phantasm,0.189


In [11]:
get_recommendations('The Godfather')

,title,similitud
0,The Godfather: Part II,0.476
1,The Godfather Trilogy: 1972-1990,0.356
2,The Godfather: Part III,0.183
3,Blood Ties,0.154
4,Household Saints,0.141
5,Start Liquidation,0.134
6,Election,0.132
7,A Mother Should Be Loved,0.130
8,Short Sharp Shock,0.130
9,Beck 28 - Familjen,0.128


**Lectura:** encuentra películas con tramas parecidas, pero se queda en lo literal: para *The Dark Knight Rises* devuelve casi puras películas de **Batman**
(porque la sinopsis dice "Batman", "Gotham"…). Quien disfrutó esa película probablemente también disfrutaría **otras de Christopher Nolan**, y eso la sinopsis no lo captura.

## 4. Recomendador basado en metadatos (director, actores, géneros, keywords)

Unimos `credits.csv` (elenco y equipo) y `keywords.csv` (palabras clave de la trama).

> Si `credits.csv` no está en `./data/`, el notebook sigue funcionando con **palabras clave y géneros**
> (sin actores ni director). Con el archivo completo se obtienen las recomendaciones por director que describe la lección.

In [12]:
import os

keywords = pd.read_csv('./data/keywords.csv')

# Filas con IDs corruptos (fechas en vez de números) en movies_metadata
df = df.drop([19730, 29503, 35587, 35803], errors='ignore')
keywords['id'] = keywords['id'].astype('int')
df['id'] = df['id'].astype('int')

HAY_CREDITS = os.path.exists('./data/credits.csv')
if HAY_CREDITS:
    credits = pd.read_csv('./data/credits.csv')
    credits['id'] = credits['id'].astype('int')
    df = df.merge(credits, on='id')
else:
    # Sin credits.csv no hay elenco ni director: se dejan vacíos y el recomendador
    # trabaja solo con palabras clave y géneros.
    print('⚠️ No se encontró ./data/credits.csv -> se omiten actores y director.')
    df['cast'] = '[]'
    df['crew'] = '[]'

df = df.merge(keywords, on='id')
# El dataset trae películas repetidas con el mismo id: dejamos una sola
df = df.drop_duplicates(subset='id').reset_index(drop=True)
print(df.shape, '| credits.csv disponible:', HAY_CREDITS)
df[['title', 'cast', 'crew', 'keywords', 'genres']].head(2)

⚠️ No se encontró ./data/credits.csv -> se omiten actores y director.


(45431, 27) | credits.csv disponible: False


,title,cast,crew,keywords,genres
0,Toy Story,[],[],"[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': '...","[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Co..."
1,Jumanji,[],[],"[{'id': 10090, 'name': 'board game'}, {'id': 10941, 'nam...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fa..."


Las columnas vienen como texto con listas "serializadas". `literal_eval` las convierte en listas de Python:

In [13]:
for feature in ['cast', 'crew', 'keywords', 'genres']:
    df[feature] = df[feature].apply(literal_eval)

df['cast'].iloc[0][:2]

[]

### Extracción de director, top 3 actores, géneros y keywords

In [14]:
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

def get_list(x):
    """Primeros 3 nombres de la lista (o todos si hay menos)."""
    if isinstance(x, list):
        names = [i['name'] for i in x]
        return names[:3]
    return []

df['director'] = df['crew'].apply(get_director)
for feature in ['cast', 'keywords', 'genres']:
    df[feature] = df[feature].apply(get_list)

df[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Toy Story,[],NaN,"[jealousy, toy, boy]","[Animation, Comedy, Family]"
1,Jumanji,[],NaN,"[board game, disappearance, based on children's book]","[Adventure, Fantasy, Family]"
2,Grumpier Old Men,[],NaN,"[fishing, best friend, duringcreditsstinger]","[Romance, Comedy]"


### Limpieza: minúsculas y sin espacios

Así **"Johnny Depp"** y **"Johnny Galecki"** se vuelven `johnnydepp` y `johnnygalecki`, y el vectorizador no los confunde por compartir "johnny".

In [15]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(' ', '')) for i in x]
    if isinstance(x, str):
        return str.lower(x.replace(' ', ''))
    return ''

for feature in ['cast', 'keywords', 'director', 'genres']:
    df[feature] = df[feature].apply(clean_data)

### Sustituto cuando no hay `credits.csv`: saga y productoras

Sin actores ni director solo quedan géneros y unas pocas palabras clave, y las recomendaciones se vuelven genéricas.
`movies_metadata.csv` trae dos metadatos que capturan parte de ese "estilo":
- **`belongs_to_collection`**: la saga (p. ej. *The Dark Knight Collection*).
- **`production_companies`**: las productoras. Muchos directores tienen la suya: *Syncopy* es de Christopher Nolan y *American Zoetrope* de Francis Ford Coppola.

Si `credits.csv` existe, esta columna queda vacía y el recomendador es exactamente el de la lección.

In [16]:
def parse_lista(x):
    try:
        v = literal_eval(x) if isinstance(x, str) else x
    except (ValueError, SyntaxError):
        return []
    return v if isinstance(v, list) else []

def get_collection(x):
    try:
        v = literal_eval(x) if isinstance(x, str) else None
    except (ValueError, SyntaxError):
        return ''
    return v['name'] if isinstance(v, dict) and 'name' in v else ''

if HAY_CREDITS:
    df['extra'] = [[] for _ in range(len(df))]
else:
    companias = df['production_companies'].apply(parse_lista).apply(get_list).apply(clean_data)
    coleccion = df['belongs_to_collection'].apply(get_collection).apply(clean_data)
    # La saga se repite 2 veces porque es una señal muy fuerte de similitud
    df['extra'] = [c + ([s, s] if s else []) for c, s in zip(companias, coleccion)]

df[['title', 'extra']].head(3)

,title,extra
0,Toy Story,"[pixaranimationstudios, toystorycollection, toystorycoll..."
1,Jumanji,"[tristarpictures, teitlerfilm, interscopecommunications]"
2,Grumpier Old Men,"[warnerbros., lancastergate, grumpyoldmencollection, gru..."


### "Metadata soup": todo en un solo texto por película

In [17]:
def create_soup(x):
    return (' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' '
            + ' '.join(x['genres']) + ' ' + ' '.join(x['extra']))

df['soup'] = df.apply(create_soup, axis=1)
df[['title', 'soup']].head(3)

,title,soup
0,Toy Story,jealousy toy boy animation comedy family pixaranimatio...
1,Jumanji,boardgame disappearance basedonchildren'sbook adventur...
2,Grumpier Old Men,fishing bestfriend duringcreditsstinger romance comedy...


### CountVectorizer en lugar de TF-IDF

Con TF-IDF, un director que dirigió muchas películas perdería peso (IDF). Aquí **no** queremos eso: que Nolan aparezca en varias películas es justo la señal que buscamos.

In [18]:
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])
print('Matriz de conteo:', count_matrix.shape)

df = df.reset_index(drop=True)
indices2 = pd.Series(df.index, index=df['title'])
indices2 = indices2[~indices2.index.duplicated(keep='first')]

def get_recommendations_meta(title, matriz=None, n=10):
    matriz = count_matrix if matriz is None else matriz
    idx = indices2[title]
    sim = cosine_similarity(matriz[idx], matriz).ravel()   # equivale a cosine_sim2[idx]
    orden = [i for i in sim.argsort()[::-1] if i != idx][:n]
    return pd.DataFrame({'title': df['title'].iloc[orden].values,
                         'director': df['director'].iloc[orden].values,
                         'similitud': sim[orden].round(3)})

get_recommendations_meta('The Dark Knight Rises')

Matriz de conteo: (45431, 31621)


,title,director,similitud
0,The Dark Knight,,0.859
1,Batman Begins,,0.788
2,American Yakuza,,0.480
3,Lockdown,,0.480
4,Desamuduru,,0.480
5,The Longest Nite,,0.480
6,Shaitan,,0.480
7,Boca,,0.480
8,Postman Blues,,0.480
9,Racketeer,,0.480


In [19]:
get_recommendations_meta('The Godfather')

,title,director,similitud
0,The Godfather: Part III,,0.727
1,The Godfather: Part II,,0.609
2,Internes Can't Take Money,,0.452
3,The Strange Affair,,0.452
4,Detective Story,,0.452
5,In the Name of the Law,,0.452
6,Union Station,,0.452
7,One on One,,0.426
8,Aprel,,0.426
9,The Browning Version,,0.426


**Lectura:** al usar metadatos estructurados (en vez de la sinopsis completa) las recomendaciones dejan de depender de palabras sueltas como "Batman".
Con `credits.csv` completo, para *The Dark Knight Rises* aparecen otras películas de **Christopher Nolan** (*The Prestige*, *Batman Begins*) y para *The Godfather* otras de **Coppola**:
los metadatos capturan el "estilo" (quién la hizo y quién actúa), no solo de qué trata.

Un problema que sí se nota: aparecen títulos muy poco conocidos y con pocos votos. Lo resolvemos en la siguiente sección.

## 5. Mejoras sugeridas

### 5.1 Más peso al director
Repetimos el nombre del director 3 veces en la "sopa", así cuenta más en la similitud.

In [20]:
def create_soup_director(x, peso=3):
    return (' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' '
            + ' '.join([x['director']] * peso) + ' ' + ' '.join(x['genres']) + ' ' + ' '.join(x['extra']))

count_dir = CountVectorizer(stop_words='english')
count_matrix_dir = count_dir.fit_transform(df.apply(create_soup_director, axis=1))

get_recommendations_meta('The Dark Knight Rises', matriz=count_matrix_dir)

,title,director,similitud
0,The Dark Knight,,0.859
1,Batman Begins,,0.788
2,American Yakuza,,0.480
3,Lockdown,,0.480
4,Desamuduru,,0.480
5,The Longest Nite,,0.480
6,Shaitan,,0.480
7,Boca,,0.480
8,Postman Blues,,0.480
9,Racketeer,,0.480


### 5.2 Filtro de popularidad (híbrido: contenido + calificación)

1. Tomamos las **30 películas más similares**.
2. Descartamos las que no tienen suficientes votos (percentil 60 de esas 30).
3. Las ordenamos por la **calificación ponderada de IMDB** y devolvemos las 10 mejores.

Así combinamos *"se parece a lo que te gustó"* con *"y además es buena"*.

In [21]:
def improved_recommendations(title, matriz=None, n=10):
    matriz = count_matrix_dir if matriz is None else matriz
    idx = indices2[title]
    sim = cosine_similarity(matriz[idx], matriz).ravel()
    top30 = [i for i in sim.argsort()[::-1] if i != idx][:30]

    cand = df.iloc[top30][['title', 'director', 'vote_count', 'vote_average']].copy()
    cand['vote_count'] = pd.to_numeric(cand['vote_count'], errors='coerce').fillna(0)
    cand['vote_average'] = pd.to_numeric(cand['vote_average'], errors='coerce').fillna(0)
    C_ = cand['vote_average'].mean()
    m_ = cand['vote_count'].quantile(0.60)

    cand = cand[cand['vote_count'] >= m_].copy()
    cand['score'] = cand.apply(weighted_rating, axis=1, m=m_, C=C_).round(3)
    return cand.sort_values('score', ascending=False).head(n).reset_index(drop=True)

improved_recommendations('The Dark Knight Rises')

,title,director,vote_count,vote_average,score
0,The Dark Knight,,12269.0,8.3,8.298
1,Batman Begins,,7511.0,7.5,7.498
2,Shaitan,,17.0,6.7,6.216
3,Jiang Hu,,8.0,6.9,6.043
4,Postman Blues,,8.0,6.6,5.893
5,The Longest Nite,,8.0,6.4,5.793
6,Island of Fire,,26.0,5.9,5.732
7,American Yakuza,,15.0,5.9,5.652
8,The Loner,,10.0,5.7,5.472
9,Murder 2,,14.0,5.6,5.450


In [22]:
improved_recommendations('The Godfather')

,title,director,vote_count,vote_average,score
0,The Godfather: Part II,,3418.0,8.3,8.294
1,The Godfather: Part III,,1589.0,7.1,7.093
2,Detective Story,,23.0,6.5,6.254
3,Cruel Winter Blues,,10.0,6.8,6.243
4,Blue Hill Avenue,,9.0,6.8,6.208
5,True Colors,,15.0,6.5,6.164
6,The Musketeers of Pig Alley,,20.0,6.1,5.928
7,A Good Lawyer's Wife,,9.0,6.3,5.923
8,Daniel,,10.0,6.2,5.886
9,Manuscripts Don't Burn,,8.0,6.1,5.789


In [23]:
improved_recommendations('Toy Story')

,title,director,vote_count,vote_average,score
0,Up,,7048.0,7.8,7.794
1,Toy Story 3,,4710.0,7.6,7.592
2,Ratatouille,,4510.0,7.5,7.493
3,Toy Story 2,,3914.0,7.3,7.293
4,Barbie as The Princess & the Pauper,,193.0,6.9,6.805
5,Toy Story That Time Forgot,,249.0,6.8,6.733
6,Dug's Special Mission,,72.0,6.9,6.683
7,Scooby-Doo! Camp Scare,,23.0,7.3,6.652
8,The Bugs Bunny/Road Runner Movie,,29.0,6.7,6.389
9,Scooby-Doo! and the Monster of Mexico,,33.0,6.3,6.173


## 6. Comparación lado a lado

In [24]:
peli = 'The Dark Knight Rises'
hibrido = list(improved_recommendations(peli)['title'])
pd.DataFrame({
    # get_recommendations conserva por defecto el catálogo original (antes del merge)
    'Sinopsis (TF-IDF)': get_recommendations(peli)['title'].values,
    'Metadatos': get_recommendations_meta(peli)['title'].values,
    'Metadatos + director x3': get_recommendations_meta(peli, matriz=count_matrix_dir)['title'].values,
    'Híbrido (contenido + popularidad)': hibrido + [''] * (10 - len(hibrido)),
})

,Sinopsis (TF-IDF),Metadatos,Metadatos + director x3,Híbrido (contenido + popularidad)
0,The Dark Knight,The Dark Knight,The Dark Knight,The Dark Knight
1,Batman Forever,Batman Begins,Batman Begins,Batman Begins
2,Batman Returns,American Yakuza,American Yakuza,Shaitan
3,Batman: Under the Red Hood,Lockdown,Lockdown,Jiang Hu
4,Batman,Desamuduru,Desamuduru,Postman Blues
5,Batman Unmasked: The Psychology of the Dark Knight,The Longest Nite,The Longest Nite,The Longest Nite
6,Batman Beyond: Return of the Joker,Shaitan,Shaitan,Island of Fire
7,Batman: Year One,Boca,Boca,American Yakuza
8,"Batman: The Dark Knight Returns, Part 1",Postman Blues,Postman Blues,The Loner
9,Batman: Mask of the Phantasm,Racketeer,Racketeer,Murder 2


## 7. Conclusiones

| Recomendador | Ventaja | Limitación |
|---|---|---|
| **Simple (IMDB)** | Fácil, sin datos del usuario, ideal para usuarios nuevos (*cold start*) | Misma lista para todos |
| **Sinopsis (TF-IDF)** | Encuentra tramas parecidas | Muy literal (todo Batman) |
| **Metadatos** | Captura director/actores/estilo | Recomienda títulos poco conocidos |
| **Híbrido** | Similar **y** bien calificada | Sigue sin conocer los gustos reales del usuario |

**Siguiente paso natural:** *filtrado colaborativo*, que usa las calificaciones de usuarios parecidos a ti
("gente que vio esto también vio…") y no necesita metadatos. Es la base de lo que usan Netflix o Spotify.

**Nota técnica:** calcular la similitud solo para la película consultada (en vez de la matriz completa de 45k × 45k)
da el mismo resultado y baja el uso de memoria de ~16 GB a menos de 1 MB por consulta.